# Pre-entrega 6 -- Orquestador multi-agente especializado
## Demo de los 3 escenarios de delegación

Este notebook es el reemplazo del video pedido en la consigna (no es posible
grabar pantalla dentro de este entorno de ejecución). En vez de eso, acá se
ve el sistema completo corriendo **con un LLM real**, mostrando paso a paso
cómo el Supervisor va delegando tareas entre el Investigador y el Analista.

**Dominio elegido:** investigación y análisis de sentimiento de noticias de
Inteligencia Artificial (noticias simuladas en memoria, sin ninguna API de
búsqueda externa -- ver `agents/research_agent.py`).

**Qué hay que tener antes de correr esto:**
- Un archivo `.env` en esta carpeta con `OPENAI_API_KEY` (o `ANTHROPIC_API_KEY`
  si cambiás `LLM_PROVIDER=anthropic`). Ver `.env.example`.
- Las dependencias de `requirements.txt` instaladas.

Los 3 escenarios que se muestran son los mismos que cubre la suite de tests
automatizados en `tests/` (ahí están probados con un LLM *falso*, para que
no dependan de ninguna API ni cuesten dinero en cada corrida de CI). Acá, en
cambio, el Supervisor decide con un LLM de verdad.

In [ ]:
import asyncio
import json

from dotenv import load_dotenv

from graph import construir_grafo, ejecutar_orquestador

load_dotenv(override=True)

### Una ayuda visual: ver la traza paso a paso

`ejecutar_orquestador()` devuelve solamente el estado FINAL del grafo. Para
poder ver, en esta demo, quién actuó en cada paso (como si fuera el video que
no podemos grabar), usamos `grafo.astream(..., stream_mode="updates")`, que
va emitiendo un evento por cada nodo que se ejecuta.

In [ ]:
async def ejecutar_con_traza(grafo, topic: str) -> dict:
    """Igual que `ejecutar_orquestador`, pero imprimiendo en vivo qué
    nodo se ejecuta en cada paso -- para poder "ver" la delegación."""
    estado: dict = {
        "messages": [],
        "topic": topic,
        "next_agent": "",
        "task_completed": False,
        "pasos": 0,
        "investigacion": None,
        "analisis": None,
    }
    config = {"recursion_limit": 50}

    async for evento in grafo.astream(estado, config=config, stream_mode="updates"):
        for nodo, salida in evento.items():
            print(f"-> Nodo '{nodo}' se ejecutó. Actualizó: {list(salida.keys())}")
            if nodo == "Supervisor":
                print(f"   Decisión del Supervisor: {salida.get('next_agent')!r} (paso {salida.get('pasos')})")
            if nodo == "Investigador":
                inv = salida["investigacion"]
                print(f"   Investigador buscó '{inv.tema_buscado}' -> {len(inv.noticias)} noticia(s) encontradas.")
            if nodo == "Analista":
                an = salida["analisis"]
                print(f"   Analista concluyó: sentimiento_general={an.sentimiento_general!r}, "
                      f"score_promedio={an.score_promedio}, suficiente={an.suficiente}")
            estado.update(salida)

    return estado

## Escenario 1: camino feliz

El Supervisor manda primero al Investigador con un tema que sí tiene
noticias simuladas disponibles, y después al Analista. Como el análisis
resulta "suficiente" (sí había noticias), el Supervisor finaliza sin
necesidad de ningún refinamiento.

In [ ]:
# "modelos" matchea la etiqueta "modelos" de 2 noticias simuladas (ver _NOTICIAS_IA).
grafo_1 = construir_grafo()  # usa el LLM real (según LLM_PROVIDER en .env)
resultado_1 = await ejecutar_con_traza(grafo_1, "modelos")

print("\n=== RESULTADO FINAL (escenario 1) ===")
print("Pasos totales:", resultado_1["pasos"])
print("Noticias encontradas:", len(resultado_1["investigacion"].noticias))
print("Sentimiento general:", resultado_1["analisis"].sentimiento_general)
print("¿Tarea completada?:", resultado_1["task_completed"])

## Escenario 2: ciclo de refinamiento

Acá le pedimos al orquestador un tema deliberadamente demasiado específico
(`"blockchain cuántico"`), que no matchea ninguna noticia simulada. El
Analista recibe 0 noticias y marca el análisis como **insuficiente**
(`suficiente=False`). El Supervisor, en vez de rendirse, manda de nuevo al
Investigador -- que detecta que el intento anterior fue insuficiente y
amplía la búsqueda al término genérico `"ia"`, que sí encuentra resultados.
Recién ahí el Analista puede completar un análisis suficiente y el
Supervisor finaliza.

Esto es la demostración concreta de **evitar la "Contaminación de
Contexto"**: el Investigador no necesita leer ningún mensaje del Analista
para saber que tiene que refinar -- lo decide mirando el campo estructurado
`analisis.suficiente` del estado compartido (ver `investigador_node` en
`graph.py`).

In [ ]:
grafo_2 = construir_grafo()
resultado_2 = await ejecutar_con_traza(grafo_2, "blockchain cuántico")

print("\n=== RESULTADO FINAL (escenario 2) ===")
print("Pasos totales:", resultado_2["pasos"])
print("Tema de la búsqueda final:", resultado_2["investigacion"].tema_buscado)
print("Noticias encontradas (tras refinar):", len(resultado_2["investigacion"].noticias))
print("¿Tarea completada?:", resultado_2["task_completed"])

## Escenario 3: corte por contador de pasos ("Supervisor Infinito")

Para demostrar la protección contra el otro error común que señala la
consigna, simulamos un Supervisor "roto" que **siempre** pide más
investigación, sin importar el estado (`decisor_terco`, definido más abajo).
Sin ninguna protección, esto haría que el grafo se cuelgue para siempre. Con
el contador `pasos` y el tope `max_pasos` (ver `supervisor_node` en
`graph.py`), el grafo corta solo y termina igual, aunque sin un análisis
"real" -- priorizando terminar por sobre colgarse.

Este escenario inyecta la función de decisión directamente (no usa el LLM
real para el Supervisor), así que no hace falta ninguna API key para esta
parte -- aunque el Investigador sí sigue usando el LLM real para decidir
cómo llamar a su herramienta.

In [ ]:
async def decisor_terco(state):
    """Un Supervisor 'roto': siempre quiere más investigación, nunca
    decide pasar al Analista ni finalizar por su cuenta."""
    return "Investigador"


grafo_3 = construir_grafo(decidir_siguiente_paso=decisor_terco, max_pasos=3)
resultado_3 = await ejecutar_con_traza(grafo_3, "un tema cualquiera")

print("\n=== RESULTADO FINAL (escenario 3) ===")
print("Pasos totales:", resultado_3["pasos"])
print("¿Tarea completada?:", resultado_3["task_completed"])
print("(El grafo terminó solo por el contador de pasos, en vez de colgarse en un ciclo infinito.)")

## Conclusión

Los 3 escenarios muestran, con un LLM real tomando las decisiones del
Supervisor (salvo en el escenario 3, donde se inyecta deliberadamente un
decisor roto para poder *forzar* el error que se quiere prevenir):

1. **Arquitectura y flujo del grafo:** topología jerárquica con un
   Supervisor que rutea dinámicamente hacia especialistas, usando aristas
   condicionales tipadas (`Literal["Investigador", "Analista", "FINALIZAR"]`).
2. **Especialización de agentes y herramientas:** cada especialista tiene
   una sola herramienta disponible y un prompt que lo limita a su tarea.
3. **Gestión de estado y validación:** toda la información que circula
   (`ResultadoInvestigacion`, `ResultadoAnalisis`) son modelos Pydantic
   validados, no texto libre.
4. **Los 2 errores comunes a evitar** quedan resueltos con mecanismos
   concretos en el código, no solo con instrucciones de prompt: el contador
   de pasos (`pasos` / `MAX_PASOS`) evita el Supervisor Infinito, y el pasar
   solo instrucciones puntuales + resúmenes de estado estructurado (nunca el
   historial completo de mensajes) evita la Contaminación de Contexto.